## Data Cleaning – start cleaning raw data. Handle missing values, fix date formats

Import libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder



Load data

In [2]:
root = Path().resolve().parents[0] if Path().resolve().name == "notebooks" else Path().resolve()
dfr = root / "data" / "raw" / "data.csv"
df = pd.read_csv(dfr, sep=';', low_memory=False)
df.columns

Index(['COMPANY_ID', 'CONTACT_NAME', 'COMPANY_NAME', 'VEEQO_PRODUCT',
       'ACCOUNT_OWNER_NAME', 'COUNTRY', 'SIGNUP_TYPE', 'EMAIL', 'DECILE',
       'SELLER_SIZE',
       ...
       'SHIPMENTS_BUY_SHIPPING_DHL', 'SHIPMENTS_BUY_SHIPPING_SWA',
       'SHIPMENTS_BUY_SHIPPING_OTHER', 'SHIPMENTS_UPS', 'SHIPMENTS_USPS',
       'SHIPMENTS_FEDEX', 'SHIPMENTS_DHL', 'SHIPMENTS_SWA', 'SHIPMENTS_OTHER',
       'ADOBE_ECID'],
      dtype='object', length=124)

In [3]:
df.head()

,COMPANY_ID,CONTACT_NAME,COMPANY_NAME,VEEQO_PRODUCT,ACCOUNT_OWNER_NAME,COUNTRY,SIGNUP_TYPE,EMAIL,DECILE,SELLER_SIZE,...,SHIPMENTS_BUY_SHIPPING_DHL,SHIPMENTS_BUY_SHIPPING_SWA,SHIPMENTS_BUY_SHIPPING_OTHER,SHIPMENTS_UPS,SHIPMENTS_USPS,SHIPMENTS_FEDEX,SHIPMENTS_DHL,SHIPMENTS_SWA,SHIPMENTS_OTHER,ADOBE_ECID
0,263028,King,Milu Network Tech,new product,Ivan Betanzos,US,Amazon SSO,roman3325@sina.com,9.0,Small,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17655601342629931110826685221210694887
1,102953,Arvind Aswani,Sporting Genius Ltd,new product,Josh Packman,UK,Email Sign-up,info@aswanisports.com,NaN,Small,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,91326,Yasir,US Prime Outlet,new product,App Flow User,US,Amazon SSO,myasir026@gmail.com,8.0,Medium,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,138633,NaN,Luxizo Depot LLC,new product,Ivan Betanzos,US,Amazon SSO,nehapatel91@icloud.com,7.0,Medium,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,258729,Todoroff,Cape Cod Challenger Club,new product,Ivan Betanzos,US,Amazon SSO,andrew@capecodchallenger.org,9.0,Small,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Simple check for duplicates and missing values

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df.isnull().sum()

COMPANY_ID                0
CONTACT_NAME          14548
COMPANY_NAME             24
VEEQO_PRODUCT             0
ACCOUNT_OWNER_NAME     6931
                      ...  
SHIPMENTS_FEDEX       83986
SHIPMENTS_DHL         83975
SHIPMENTS_SWA         83953
SHIPMENTS_OTHER       83227
ADOBE_ECID            77643
Length: 124, dtype: int64

Convert Russian month names to English and then to datetime

In [6]:
ru_month_map = {
    'январь': 'Jan', 'января': 'Jan', 'янв': 'Jan',
    'февраль': 'Feb', 'февраля': 'Feb', 'фев': 'Feb',
    'март': 'Mar', 'марта': 'Mar', 'мар': 'Mar',
    'апрель': 'Apr', 'апреля': 'Apr', 'апр': 'Apr',
    'май': 'May', 'мая': 'May',
    'июнь': 'Jun', 'июня': 'Jun', 'июн': 'Jun',
    'июль': 'Jul', 'июля': 'Jul', 'июл': 'Jul',
    'август': 'Aug', 'августа': 'Aug', 'авг': 'Aug',
    'сентябрь': 'Sep', 'сентября': 'Sep', 'сен': 'Sep',
    'октябрь': 'Oct', 'октября': 'Oct', 'окт': 'Oct',
    'ноябрь': 'Nov', 'ноября': 'Nov', 'ноя': 'Nov',
    'декабрь': 'Dec', 'декабря': 'Dec', 'дек': 'Dec'
}

date_cols = [
    col for col in df.columns
    if any(k in col.upper() for k in ['DATE', 'AT', 'LOGIN', 'INTERACTION', 'CREATED', 'UPDATED', 'LAUNCHED', 'FIRST', 'LAST', 'WEEK'])
    and not any(e in col.upper() for e in ['TIME_TO', 'DAYS_', 'HOURS_', 'DURATION'])
]

for col in date_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).replace(ru_month_map, regex=True)
        df[col] = pd.to_datetime(df[col], errors='coerce')
        df[col] = df[col].fillna(pd.Timestamp('2099-01-01'))

duration_cols = [col for col in df.columns if 'TIME_TO' in col.upper() or 'DAYS_' in col.upper()]
for col in duration_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

C:\Temp\ipykernel_15932\2825061153.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')
C:\Temp\ipykernel_15932\2825061153.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')
C:\Temp\ipykernel_15932\2825061153.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')
C:\Temp\ipykernel_15932\2825061153.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure p

In [7]:
df.head()

,COMPANY_ID,CONTACT_NAME,COMPANY_NAME,VEEQO_PRODUCT,ACCOUNT_OWNER_NAME,COUNTRY,SIGNUP_TYPE,EMAIL,DECILE,SELLER_SIZE,...,SHIPMENTS_BUY_SHIPPING_DHL,SHIPMENTS_BUY_SHIPPING_SWA,SHIPMENTS_BUY_SHIPPING_OTHER,SHIPMENTS_UPS,SHIPMENTS_USPS,SHIPMENTS_FEDEX,SHIPMENTS_DHL,SHIPMENTS_SWA,SHIPMENTS_OTHER,ADOBE_ECID
0,263028,King,Milu Network Tech,new product,Ivan Betanzos,US,Amazon SSO,roman3325@sina.com,9.0,Small,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17655601342629931110826685221210694887
1,102953,Arvind Aswani,Sporting Genius Ltd,new product,Josh Packman,UK,Email Sign-up,info@aswanisports.com,NaN,Small,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,91326,Yasir,US Prime Outlet,new product,App Flow User,US,Amazon SSO,myasir026@gmail.com,8.0,Medium,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,138633,NaN,Luxizo Depot LLC,new product,Ivan Betanzos,US,Amazon SSO,nehapatel91@icloud.com,7.0,Medium,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,258729,Todoroff,Cape Cod Challenger Club,new product,Ivan Betanzos,US,Amazon SSO,andrew@capecodchallenger.org,9.0,Small,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Fill missing values

In [8]:
datetime_cols = date_cols 

pct_cols = [c for c in df.columns if 'PERCENT' in c or 'WOW' in c or 'MOM' in c]
for c in pct_cols:
    if c in df.columns and df[c].dtype == object:
        df[c] = (
            df[c].astype(str)
            .str.replace('%', '', regex=False)
            .str.replace(' ', '', regex=False)
            .str.replace(',', '.', regex=False)
        )
        df[c] = pd.to_numeric(df[c], errors='coerce')

structural_zero_cols = [
    col for col in df.columns if any(
        k in col for k in [
            'SHIPMENT', 'ORDER', 'CALL', 'DEMO', 'EMAIL', 'USER',
            'WAREHOUSE', 'CHANNEL', 'RE_EMAIL', 'DHL', 'AMAZON',
            'EBAY', 'SHOPIFY', 'COUNT', 'L28'
        ]
    ) and col not in datetime_cols
]

for col in structural_zero_cols:
    if col in df.columns:
        if df[col].dtype == object:
            df[col] = (
                df[col].astype(str)
                .str.replace(' ', '', regex=False)
                .str.replace(',', '.', regex=False)
            )
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(0)

In [9]:
median_cols = [
    c for c in df.select_dtypes(include=[np.number]).columns 
    if c not in structural_zero_cols 
    and c not in ['COMPANY_ID'] 
    and df[c].isnull().sum() > 0
]
for col in median_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
df.isnull().sum()

COMPANY_ID                0
CONTACT_NAME          14548
COMPANY_NAME             24
VEEQO_PRODUCT             0
ACCOUNT_OWNER_NAME        0
                      ...  
SHIPMENTS_FEDEX           0
SHIPMENTS_DHL             0
SHIPMENTS_SWA             0
SHIPMENTS_OTHER           0
ADOBE_ECID            77643
Length: 124, dtype: int64

In [10]:
date_cols = df.select_dtypes(include=['datetime64[ns]']).columns
df[date_cols] = df[date_cols].fillna(pd.Timestamp('2099-01-01'))

num_cols = df.select_dtypes(include=['number']).columns
df[num_cols] = df[num_cols].fillna(0)

bool_cols = df.select_dtypes(include=['bool']).columns
df[bool_cols] = df[bool_cols].fillna(False)

obj_cols = df.select_dtypes(include=['object']).columns
df[obj_cols] = df[obj_cols].fillna('Unknown')
df.isnull().sum()



COMPANY_ID            0
CONTACT_NAME          0
COMPANY_NAME          0
VEEQO_PRODUCT         0
ACCOUNT_OWNER_NAME    0
                     ..
SHIPMENTS_FEDEX       0
SHIPMENTS_DHL         0
SHIPMENTS_SWA         0
SHIPMENTS_OTHER       0
ADOBE_ECID            0
Length: 124, dtype: int64

In [11]:
df.head(10)

,COMPANY_ID,CONTACT_NAME,COMPANY_NAME,VEEQO_PRODUCT,ACCOUNT_OWNER_NAME,COUNTRY,SIGNUP_TYPE,EMAIL,DECILE,SELLER_SIZE,...,SHIPMENTS_BUY_SHIPPING_DHL,SHIPMENTS_BUY_SHIPPING_SWA,SHIPMENTS_BUY_SHIPPING_OTHER,SHIPMENTS_UPS,SHIPMENTS_USPS,SHIPMENTS_FEDEX,SHIPMENTS_DHL,SHIPMENTS_SWA,SHIPMENTS_OTHER,ADOBE_ECID
0,263028,King,Milu Network Tech,new product,0.0,0.0,Amazon SSO,0.0,9.0,Small,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17655601342629931110826685221210694887
1,102953,Arvind Aswani,Sporting Genius Ltd,new product,0.0,0.0,Email Sign-up,0.0,9.0,Small,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown
2,91326,Yasir,US Prime Outlet,new product,0.0,0.0,Amazon SSO,0.0,8.0,Medium,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown
3,138633,Unknown,Luxizo Depot LLC,new product,0.0,0.0,Amazon SSO,0.0,7.0,Medium,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown
4,258729,Todoroff,Cape Cod Challenger Club,new product,0.0,0.0,Amazon SSO,0.0,9.0,Small,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown
5,272042,Haddad,Health Mobius LLC,new product,0.0,0.0,Email Sign-up,0.0,9.0,Small,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,84559775290099120164050692269234574670
6,142210,Auble,"Goki America, Inc.",new product,0.0,0.0,Email Sign-up,0.0,9.0,Small,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown
7,259531,Unknown,Andy,new product,0.0,0.0,Amazon SSO,0.0,9.0,Small,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown
8,80728,Ad Majoram,Ad Majoram Dei Gloriam Shop,new product,0.0,0.0,Amazon SSO,0.0,9.0,Small,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown
9,67347,Christian Ha,Christian Ha,new product,0.0,0.0,Email Sign-up,0.0,9.0,Small,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown


Save cleaned data

In [12]:
data_cleaned = df.to_csv(root / "data" / "processed" / "cleaned_data.csv", index=False)

## All missing values filled, dates standardized
